# prefkit analysis (CPU)

Load `results/*.json` with `backend==hf` only. Reject mixing ollama+hf. E3 plot: original Qwen3 sizes only.

In [ ]:
import json
from pathlib import Path

from prefkit.cms import CMSError, cms, disagreement_cards, missingness
from prefkit.outcomes import load_outcomes, id_order

ROOT = Path(".").resolve()
blobs = []
for p in sorted((ROOT / "results").glob("*.json")):
    if p.name.startswith("debug_ollama_"):
        continue
    b = json.loads(p.read_text())
    if b.get("backend") != "hf":
        continue  # skip ollama debug; do not mix into CMS/E3
    b["_path"] = str(p)
    blobs.append(b)
hf = blobs
print("hf files", len(hf))

In [ ]:
E3_HF = {
    "Qwen/Qwen3-1.7B": 1.7,
    "Qwen/Qwen3-4B": 4.0,
    "Qwen/Qwen3-8B": 8.0,
    "Qwen/Qwen3-14B": 14.0,
}
ids = id_order(load_outcomes("data/outcomes.json"))
rows = []
for b in hf:
    scores = b["scores"]
    miss = missingness(scores, ids) if set(ids) <= set(next(iter(scores.values()))) else missingness(scores, list(next(iter(scores.values()))))
    rec = {"model": b["model"], "slot": b.get("slot"), "missingness": miss, "cms": None, "error": None}
    high_miss = any(v > 20 for v in miss.values())
    try:
        if high_miss:
            rec["error"] = "missingness>20%"
        else:
            out = cms(scores, list(scores["M1"]))
            rec["cms"] = out["cms"]
            rec["matrix"] = {f"{a}|{b_}": v for (a, b_), v in out["matrix"].items()}
            rec["ranks"] = out["ranks"]
            rec["disagreement"] = disagreement_cards(out["ranks"], list(scores["M1"]))
    except CMSError as e:
        rec["error"] = str(e)
    rows.append(rec)
    print(rec["model"], rec["cms"], rec["error"], rec["missingness"])

In [ ]:
import matplotlib.pyplot as plt

xs, ys, labels = [], [], []
for rec in rows:
    if rec["model"] not in E3_HF:
        continue
    if rec["cms"] is None:
        continue
    xs.append(E3_HF[rec["model"]])
    ys.append(rec["cms"])
    labels.append(rec["model"])
if xs:
    plt.figure()
    plt.plot(xs, ys, "o-")
    plt.xlabel("params (B)")
    plt.ylabel("CMS")
    plt.title("E3 CMS vs original Qwen3 size (hf only)")
    Path("figures").mkdir(exist_ok=True)
    plt.savefig("figures/e3_cms_vs_size.png")
    plt.show()
else:
    print("no E3 CMS points yet")